## Stage 1: Setup & Data Loading

Mounts Google Drive and loads the four training files (Source1, Source2, Source3, Ground Truth). Cleans entity IDs and filters ground truth to only rows referencing valid Source1 entities.

In [ ]:
# Mount Drive and load challenge data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

base = "/content/student_resource/student_resource/dataset"

train1 = pd.read_csv(f"{base}/train/train_source1.tsv", sep="\t")
train2 = pd.read_csv(f"{base}/train/train_source2.tsv", sep="\t")
train3 = pd.read_csv(f"{base}/train/train_source3.tsv", sep="\t")
ground_truth = pd.read_csv(f"{base}/train/train_ground_truth.tsv", sep="\t")

for df in (train1, train2, train3):
    df["entity_id"] = df["entity_id"].astype(str).str.strip()
ground_truth["source1_entity_id"] = ground_truth["source1_entity_id"].astype(str).str.strip()
ground_truth = ground_truth[ground_truth["source1_entity_id"].isin(set(train1["entity_id"]))].reset_index(drop=True)

print(f"Source1: {len(train1)}  Source2: {len(train2)}  Source3: {len(train3)}  Ground truth: {len(ground_truth)}")

## Stage 2: Normalization & Blocking

Defines text normalization (lowercasing, accent-stripping for multilingual/French support, legal-suffix removal) and builds an inverted index (token → entity_id, per country) for efficient candidate generation — avoiding brute-force O(n×m) comparison across millions of records.

In [ ]:
import re
import unicodedata
from collections import defaultdict

LEGAL_SUFFIXES = {"pvt","private","ltd","limited","llc","inc","incorporated","corp",
    "corporation","co","company","llp","gmbh","plc","enterprises","enterprise",
    "group","holdings","industries","international","intl"}

def strip_accents(text):
    """Handle diacritics for open-set country generalization (e.g. French names)."""
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

def normalize_name(name):
    if pd.isna(name): return []
    name = strip_accents(str(name).lower())
    name = re.sub(r"[^\w\s]", " ", name)
    return [t for t in name.split() if t not in LEGAL_SUFFIXES and len(t) >= 2]

def normalize_address(addr):
    if pd.isna(addr): return []
    addr = strip_accents(str(addr).lower())
    addr = re.sub(r"[^\w\s]", " ", addr)
    return [t for t in addr.split() if len(t) >= 3]

def clean_str(x):
    if pd.isna(x): return ""
    return strip_accents(str(x).lower())

MAX_POSTING_LIST = 1000   # cap overly common tokens for blocking efficiency
TOP_K_CANDIDATES = 10

def build_index_multi(df):
    """Inverted index: country -> token -> [entity_ids], for name and address tokens."""
    name_idx = defaultdict(lambda: defaultdict(list))
    addr_idx = defaultdict(lambda: defaultdict(list))
    for row in df.itertuples(index=False):
        for t in set(normalize_name(row.business_name)):
            name_idx[row.country][t].append(row.entity_id)
        for t in set(normalize_address(row.business_address)):
            addr_idx[row.country][t].append(row.entity_id)
    for idx in (name_idx, addr_idx):
        for country in idx:
            for t in list(idx[country].keys()):
                if len(idx[country][t]) > MAX_POSTING_LIST:
                    del idx[country][t]
    return name_idx, addr_idx

def get_scored_candidates(name, addr, country, name_idx, addr_idx):
    """Score candidates by weighted token overlap (name tokens worth more), keep top-K."""
    scores = defaultdict(int)
    for t in normalize_name(name):
        for eid in name_idx[country].get(t, []):
            scores[eid] += 2
    for t in normalize_address(addr):
        for eid in addr_idx[country].get(t, []):
            scores[eid] += 1
    if not scores:
        return []
    return [eid for eid, _ in sorted(scores.items(), key=lambda x: -x[1])[:TOP_K_CANDIDATES]]

## Stage 3: Candidate Generation & Similarity Scoring (Training)

Generates candidate pairs for each Source1 entity via blocking, then scores each pair using a weighted combination of name and address fuzzy similarity (RapidFuzz). Writes results to disk in chunks to stay memory-safe at scale (~42M candidate pairs).

In [ ]:
!pip install rapidfuzz --quiet
from rapidfuzz import fuzz
import gc, os, time
from tqdm import tqdm

name_idx2, addr_idx2 = build_index_multi(train2)
name_idx3, addr_idx3 = build_index_multi(train3)

lookup2 = train2.set_index("entity_id")[["business_name","business_address"]].to_dict("index")
lookup3 = train3.set_index("entity_id")[["business_name","business_address"]].to_dict("index")
gt_lookup = dict(zip(ground_truth["source1_entity_id"], ground_truth["matched_entity_ids"]))

def score_pair(s1_name, s1_addr, cand_id, lookup2, lookup3):
    tgt = lookup2.get(cand_id) or lookup3.get(cand_id)
    if tgt is None:
        return None
    name_sim = fuzz.token_sort_ratio(clean_str(s1_name), clean_str(tgt["business_name"])) / 100
    addr_sim = fuzz.token_sort_ratio(clean_str(s1_addr), clean_str(tgt["business_address"])) / 100
    return 0.7 * name_sim + 0.3 * addr_sim   # name weighted higher than address

# Write candidate pairs to disk in chunks (memory-safe for millions of rows)
os.makedirs("/content/checkpoints", exist_ok=True)
OUTPUT_PATH = "/content/checkpoints/train_pairs.csv"
if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)

CHUNK_SIZE = 50000
buffer, header_written, total_written = [], False, 0

for row in tqdm(train1.itertuples(index=False), total=len(train1)):
    cands2 = get_scored_candidates(row.business_name, row.business_address, row.country, name_idx2, addr_idx2)
    cands3 = get_scored_candidates(row.business_name, row.business_address, row.country, name_idx3, addr_idx3)
    raw = gt_lookup.get(row.entity_id, "")
    true_matches = set(raw.split(",")) if isinstance(raw, str) and raw else set()

    for cid in set(cands2) | set(cands3):
        sim = score_pair(row.business_name, row.business_address, cid, lookup2, lookup3)
        if sim is not None:
            buffer.append((row.entity_id, cid, sim, 1 if cid in true_matches else 0))

    if len(buffer) >= CHUNK_SIZE:
        pd.DataFrame(buffer, columns=["s1_id","cand_id","sim","label"]).to_csv(
            OUTPUT_PATH, mode="a", header=not header_written, index=False)
        header_written, total_written, buffer = True, total_written + len(buffer), []
        gc.collect()

if buffer:
    pd.DataFrame(buffer, columns=["s1_id","cand_id","sim","label"]).to_csv(
        OUTPUT_PATH, mode="a", header=not header_written, index=False)

print(f"Total candidate pairs generated: {total_written}")

## Stage 4: Threshold Tuning (Macro F0.5)

Implements the competition's exact evaluation metric — F0.5 computed per-entity, then macro-averaged across all Source1 entities, with singleton (no-match) cases scored explicitly. Sweeps thresholds to find the one maximizing macro F0.5, since this metric is precision-heavy (false merges penalized more than missed matches).

In [ ]:
import numpy as np

# F0.5 = (1.25 * P * R) / (0.25 * P + R), macro-averaged per Source1 entity
def score_entity(true_matches, predicted):
    if not true_matches:
        return 1.0 if not predicted else 0.0   # singleton correctly identified
    if not predicted:
        return 0.0
    tp = len(true_matches & predicted)
    precision = tp / len(predicted)
    recall = tp / len(true_matches)
    denom = 0.25 * precision + recall
    return (1.25 * precision * recall) / denom if denom > 0 else 0.0

thresholds = np.arange(0.50, 0.96, 0.02)
sum_f05 = {t: 0.0 for t in thresholds}

for s1_id, group in pd.read_csv(OUTPUT_PATH).groupby("s1_id"):
    raw = gt_lookup.get(s1_id, "")
    true_matches = set(raw.split(",")) if isinstance(raw, str) and raw else set()
    for t in thresholds:
        predicted = set(group.loc[group["sim"] >= t, "cand_id"])
        sum_f05[t] += score_entity(true_matches, predicted)

best_t, best_score = max(
    ((t, sum_f05[t] / len(train1)) for t in thresholds), key=lambda x: x[1]
)
print(f"Best threshold: {best_t:.2f}  Macro F0.5: {best_score:.4f}")

In [ ]:
BEST_THRESHOLD = 0.76  # from Stage 4 tuning

test1 = pd.read_csv(f"{base}/test/test_source1.tsv", sep="\t")
test2 = pd.read_csv(f"{base}/test/test_source2.tsv", sep="\t")
test3 = pd.read_csv(f"{base}/test/test_source3.tsv", sep="\t")
for df in (test1, test2, test3):
    df["entity_id"] = df["entity_id"].astype(str).str.strip()

test_name_idx2, test_addr_idx2 = build_index_multi(test2)
test_name_idx3, test_addr_idx3 = build_index_multi(test3)
test_lookup2 = test2.set_index("entity_id")[["business_name","business_address"]].to_dict("index")
test_lookup3 = test3.set_index("entity_id")[["business_name","business_address"]].to_dict("index")

os.makedirs("/content/output", exist_ok=True)
cand_rows, match_rows = [], []

for row in tqdm(test1.itertuples(index=False), total=len(test1)):
    cands2 = get_scored_candidates(row.business_name, row.business_address, row.country, test_name_idx2, test_addr_idx2)
    cands3 = get_scored_candidates(row.business_name, row.business_address, row.country, test_name_idx3, test_addr_idx3)
    scored = []
    for cid in set(cands2) | set(cands3):
        sim = score_pair(row.business_name, row.business_address, cid, test_lookup2, test_lookup3)
        if sim is not None:
            scored.append((cid, sim))
    all_ids = list(dict.fromkeys(cid for cid, _ in scored))
    cand_rows.append((row.entity_id, ",".join(all_ids)))
    matched = list(dict.fromkeys(cid for cid, sim in scored if sim >= BEST_THRESHOLD))
    match_rows.append((row.entity_id, ",".join(matched)))

pd.DataFrame(cand_rows, columns=["source1_entity_id","candidate_entity_ids"]).to_csv(
    "/content/output/candidate_pairs.tsv", sep="\t", index=False)
pd.DataFrame(match_rows, columns=["source1_entity_id","matched_entity_ids"]).to_csv(
    "/content/output/matching_results.tsv", sep="\t", index=False)

print("Submission files generated.")

## Stage 5: Apply to Test Set & Generate Submission Files

Applies the identical normalization, blocking, and scoring pipeline to the test data (which includes an unseen country — France — testing open-set generalization) using the tuned threshold. Outputs `matching_results.tsv` and `candidate_pairs.tsv` in the exact format required by the competition validator.